# Import

In [260]:
import yfinance as yf
from datetime import datetime
import numpy as np
import pandas as pd

import sys, importlib, os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.utils
from torch.utils.data import DataLoader
from typing import Tuple
import yaml
from pathlib import Path
from datetime import datetime

from torchvision import datasets, transforms
from torch.utils.data import Subset

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

import pytorch_lightning as pl

In [261]:
# Importing project packages
cwd = Path(os.getcwd())
root_dir = cwd.parent

utils_dir = root_dir.joinpath('src','utils')
model_dir = root_dir.joinpath('src','model')

sys.path.append(str(root_dir))

In [262]:
from src.utils import WIP_SDE, WIP_processes
from src.models import components




importlib.reload(WIP_SDE)
importlib.reload(WIP_processes)
importlib.reload(components)

from src.utils.WIP_SDE import VESDE, SubVPSDE, VPSDE
from src.utils.WIP_processes import Diffusion_Processes
from src.training.ldm_module import LDMLightningModule
# from src.models.components import SinusoidalPositionEmbeddings # Required by unet_model
from src.models.UNet import UNet

importlib.reload(sys.modules['src.training.ldm_module'])
importlib.reload(sys.modules['src.models.UNet'])

device = torch.device("cpu")
# unet = UNet

In [263]:
cfg = {
    'N': 1000,
    'sde_type': 've', # or 've' / 'subvp'
    'channels': 5,    # Vital: Match your blob channel count
    # 'image_size': 32,
    'features': [128, 256, 512],
    'batch_size': 14,
    'vae_scale_factor': 0.18125,
    'learning_rate': 0.001,
    'epochs': 10,
    'channels': 5,
    'features': [128, 256, 512],
    'eps': 1e-6,
    'conditional': False
}

# Data download

In [264]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2024-12-31"

In [265]:
data = yf.download(
    tickers = ticker,
    start = start_date,
    end = end_date,
    interval = "1d",
    auto_adjust = True,
    progress = False
)

In [266]:
print(data.head())

Price          Close      High       Low      Open     Volume
Ticker          AAPL      AAPL      AAPL      AAPL       AAPL
Date                                                         
2010-01-04  6.418383  6.433078  6.369497  6.400988  493729600
2010-01-05  6.429479  6.465768  6.395589  6.436077  601904800
2010-01-06  6.327210  6.454972  6.320612  6.429479  552160000
2010-01-07  6.315515  6.358103  6.269629  6.350605  477131200
2010-01-08  6.357502  6.358102  6.269928  6.307117  447610800


# Transformation of data

In [267]:
df = data.copy()
df.reset_index(inplace=True)

In [268]:
# 1. Ensure Temporal Order
# Use the 'Date' column to sort but do not include it in the numerical tensor
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 2. Universal Anchor Transformation (Stationarity)
# We divide ALL price features at time 't' by the Close at 't-1'
prev_close = df['Close'].shift(1)
ohlc_cols = ['Open', 'High', 'Low', 'Close']

# Log-returns for prices: r_t = ln(Price_t / Close_{t-1})
ohlc_log_rets = np.log(df[ohlc_cols].div(prev_close, axis=0))

# 3. Independent Volume Processing
# Volume is a different unit; normalize it by its own previous value
volume_log_rets = np.log(df['Volume'] / df['Volume'].shift(1))

# 4. Clean and Standardize (Z-Score)
# Diffusion models require unit variance (std=1) to prevent gradient collapse
processed_df = pd.concat([ohlc_log_rets, volume_log_rets], axis=1).dropna()
data_array = processed_df.values.astype(np.float32)

# Global standardization using training set statistics
mean = data_array.mean(axis=0)
std = data_array.std(axis=0)
data_standardized = (data_array - mean) / (std + 1e-6)

print(f"Before Standardization: mean={data_array.mean():4f}, std={data_array.std():4f}")
print(f"After Standardization: mean={data_standardized.mean():4f}, std={data_standardized.std():4f}")

print("\nBefore standardization:")
for i in range(data.shape[1]):
    col_mean = data_array[:, i].mean()
    col_std = data_array[:, i].std()
    print(f"Feature {i}: mean={col_mean: .6f}, std={col_std: .6f}")

# 5. Tensor Preparation for 1D-UNet
# Shape must be (Batch, Channels, Length) -> (M, 5, 252)
N = 252 # Sequence length representing one trading year
M = len(data_standardized) // N

# Reshape and Permute: moves features from columns to channels
train_tensor = torch.tensor(data_standardized[:M*N]).view(M, N, 5).permute(0, 2, 1)

print(f"Tensor Shape: {train_tensor.shape}") # Output: torch.Size([M, 5, 252])

Before Standardization: mean=0.000392, std=0.140737
After Standardization: mean=-0.000000, std=0.999943

Before standardization:
Feature 0: mean= 0.000621, std= 0.011606
Feature 1: mean= 0.010475, std= 0.013396
Feature 2: mean=-0.009412, std= 0.015412
Feature 3: mean= 0.000972, std= 0.017558
Feature 4: mean=-0.000697, std= 0.313011
Tensor Shape: torch.Size([14, 5, 252])


# Training and Model

In [269]:
diffusion = Diffusion_Processes(cfg)

In [270]:
hparams = {
        'learning_rate': cfg['learning_rate'],
        'n_timesteps': cfg['N'],
        'batch_size': cfg['batch_size'],
        'vae_scale_factor': cfg['vae_scale_factor'],
        'is_probabilities': None
    }

In [271]:
unet = UNet(
    in_channels=cfg['channels'],   # 5 channels
    out_channels=cfg['channels'],  # 5 channels
    model_channels=64              # Adjust based on your memory constraints
)

ldm_module = LDMLightningModule(
    unet_model=unet,
    forward_process=diffusion,
    vae_encoder=None,
    vae_decoder=None,
    hparams=hparams,
    cfg=cfg
)

In [272]:
print("n_trainable =", sum(p.numel() for p in ldm_module.parameters() if p.requires_grad))

n_trainable = 8057925


In [273]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

# train_tensor: [N, ...] where N is number of samples (years/windows/etc.)
dataset = TensorDataset(train_tensor)   # each batch item is (x,)

# --- split train/val ---
val_frac = cfg.get("val_frac", 0.2)
n = len(dataset)
n_val = int(n * val_frac)
n_train = n - n_val

train_ds, val_ds = random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(cfg.get("seed", 42))
)

# --- loaders ---
batch_size = cfg["batch_size"]
num_workers = cfg.get("num_workers", 0)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,     # common for diffusion/AE training
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

# --- Lightning ---
trainer = pl.Trainer(
    max_epochs=cfg["epochs"],
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
)

trainer.fit(ldm_module, train_dataloaders=train_loader, val_dataloaders=val_loader)
print("Final training loss:", trainer.callback_metrics.get("train_loss_epoch"))
print("Final validation loss:", trainer.callback_metrics.get("val_loss"))


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | unet      | UNet    | 8.1 M  | train
1 | criterion | MSELoss | 0      | train
----------------------------------------------
8.1 M     Trainable params
0         Non-trainable params
8.1 M     Total params
32.232    Total estimated model params size (MB)
252       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\pytorch_lightning\utilities\data.py:106: Total length of `DataLoader` acro

Batch shape: [tensor([[[ 0.9345, -1.9400, -0.3747,  ...,  0.2395, -0.0967, -1.1539],
         [ 0.1868, -1.8522, -0.3721,  ..., -0.4015, -0.5869, -0.6199],
         [ 0.2678, -1.1574, -0.4646,  ..., -0.4278, -1.2425, -0.6175],
         [ 0.4341, -1.3950, -0.1349,  ..., -0.7828, -1.5907, -1.0141],
         [-0.1293, -0.1462, -0.2781,  ..., -0.1601, -0.1551,  0.0812]],

        [[ 0.0193,  0.2885,  0.1978,  ...,  0.3781, -0.8183, -0.4219],
         [-0.1480,  0.7888, -0.3526,  ...,  1.0173, -0.9470, -0.2756],
         [-0.4093,  0.4765, -0.3935,  ...,  0.7251, -0.3865,  0.1817],
         [ 0.1979,  1.1345, -0.0895,  ...,  1.3050, -0.1844,  0.2667],
         [-0.3251,  0.2689,  0.0894,  ..., -0.1679, -1.0280,  0.2841]]])]
x_start_latents shape: torch.Size([2, 5, 252])
Final training loss: None
Final validation loss: None
